# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — "What Predicts Health?" (ML Appendix)
The paper trains a Random Forest on health_score, which is itself a formula built from
position, impressions, CTR, and scroll depth. The top features it "discovers" (position 43%,
impressions 32%, scroll depth 15%) closely mirror the formula's own inputs. The paper already
flags this as descriptive, not causal — my question extends it: using the leakage-hunt pattern
from ML-05 (train with vs without the suspect feature), what would importance look like on a
target NOT built from these same fields, e.g. future health_score or clicks?

## Finding 2 — "What Predicts Growth?" (ML Appendix)
Logistic regression, 71% holdout accuracy, across 57 brands. The methodology section states an
80/20 split but doesn't specify grouped-by-brand vs random-row. My own ML-08 baseline showed a
concrete version of this risk: 0.860 Precision@50 in-sample vs ~0.64 (near base rate) once
re-evaluated on a client-held-out split. Was the paper's 71% measured on a random split that a
grouped design might reveal to be optimistic?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from getpass import getpass
import os, duckdb, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
tbl = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

raw = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {tbl}),
    decision AS (SELECT max_d - INTERVAL 30 DAY AS decision_date FROM bounds),
    prior AS (
        SELECT f.content_hash_id, ANY_VALUE(f.client_hash_id) AS client_hash_id,
            AVG(f.gsc_clicks) AS avg_daily_clicks_prior,
            AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0) AS avg_position_prior,
            COUNT(*) FILTER (WHERE f.gsc_clicks > 0) AS days_with_clicks_prior,
            SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.sessions_organic + f.sessions_ai), 0) AS ai_share_prior,
            COUNT(*) AS n_days_prior
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date - INTERVAL 90 DAY AND f.report_date < d.decision_date
        GROUP BY f.content_hash_id HAVING COUNT(*) >= 30
    ),
    future AS (
        SELECT f.content_hash_id, AVG(f.gsc_clicks) AS avg_daily_clicks_future
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date AND f.report_date < d.decision_date + INTERVAL 30 DAY
        GROUP BY f.content_hash_id
    )
    SELECT p.*, fu.avg_daily_clicks_future FROM prior p JOIN future fu USING (content_hash_id)
""").df()

raw['avg_position_prior'] = raw['avg_position_prior'].fillna(100)
raw['no_ranking_data_prior'] = (raw['avg_position_prior'] == 100).astype(int)
raw['ai_share_prior'] = raw['ai_share_prior'].fillna(0)
raw['click_consistency_prior'] = raw['days_with_clicks_prior'] / raw['n_days_prior']
raw['is_declining_future'] = (
    (raw['avg_daily_clicks_future'] < 0.75 * raw['avg_daily_clicks_prior']) &
    (raw['avg_daily_clicks_prior'] >= 1.0)
).astype(int)

eligible = raw.loc[raw['avg_daily_clicks_prior'] >= 1.0].copy()
feature_cols = ['avg_daily_clicks_prior', 'avg_position_prior', 'click_consistency_prior',
                 'ai_share_prior', 'no_ranking_data_prior']
X, y, groups = eligible[feature_cols], eligible['is_declining_future'], eligible['client_hash_id']

def precision_at_50(scores_df, label_col='is_declining_future', score_col='score'):
    return scores_df.sort_values(score_col, ascending=False)[label_col].iloc[:50].mean()

# BEFORE: naive random split
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
m_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced').fit(Xtr_r, ytr_r)
test_r = eligible.loc[Xte_r.index].copy()
test_r['score'] = m_random.predict_proba(Xte_r)[:, 1]
p50_before = precision_at_50(test_r)

# AFTER: grouped by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr_g, Xte_g, ytr_g, yte_g = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]
m_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced').fit(Xtr_g, ytr_g)
test_g = eligible.iloc[test_idx].copy()
test_g['score'] = m_grouped.predict_proba(Xte_g)[:, 1]
p50_after = precision_at_50(test_g)

print(f'BEFORE (random split)  Precision@50: {p50_before:.3f}  (test base rate: {yte_r.mean():.3f})')
print(f'AFTER  (grouped split) Precision@50: {p50_after:.3f}  (test base rate: {yte_g.mean():.3f})')
print(f'gap: {p50_before - p50_after:+.3f}')

Paste your HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (random split)  Precision@50: 1.000  (test base rate: 0.667)
AFTER  (grouped split) Precision@50: 0.740  (test base rate: 0.652)
gap: +0.260


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit — attack checklist
- [x] Timeline: all 5 features are prior-window only (no "_future" fields) — confirmed by assertion
- [x] Label-derived/sibling test: honest AUC 0.633 vs. deliberately-leaked AUC 0.997 — the leak
      test harness correctly detects a smuggled future-window field
- [x] No product flags used as features (health_score, needs_ctr_fix, etc.) — none were ever
      queried from the warehouse
- [x] Grouped split run and compared against random split (see Section 2)
- [x] Base rate printed next to every metric throughout
- [x] ga4_coverage_prior excluded — confirmed CONFOUNDED (client-level artifact) in ML-06

In [2]:
# 1. Timeline check — every feature column name should say "_prior", none "_future"
print('feature columns:', feature_cols)
assert not any('future' in c for c in feature_cols), 'FAIL: a future-window field is in X'

# 2. Label-derived / sibling check — deliberately add the smuggled field, confirm AUC jumps
from sklearn.metrics import roc_auc_score
Xl = eligible[feature_cols + ['avg_daily_clicks_future']]
Xtrl, Xtel, ytrl, ytel = train_test_split(Xl, y, test_size=0.3, random_state=42, stratify=y)
leak_auc = roc_auc_score(ytel, RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtrl, ytrl).predict_proba(Xtel)[:, 1])
honest_auc = roc_auc_score(yte_r, m_random.predict_proba(Xte_r)[:, 1])
print(f'honest AUC: {honest_auc:.3f}  |  deliberately-leaky AUC: {leak_auc:.3f}  (should jump toward 1.0)')

# 3. Product-flag check — none of these features come from an existing FlyRank rule/score
print('no product flags (health_score, needs_ctr_fix, etc.) in feature set: CONFIRMED — none were ever queried from the warehouse')

# 4. Base rate printed next to every metric — already done above (test_r/test_g base rates)

# 5. ga4_coverage_prior — explicitly excluded, confirmed CONFOUNDED in ML-06 (client-level artifact, not page-level signal)
print('ga4_coverage_prior excluded:', 'ga4_coverage_prior' not in feature_cols)

feature columns: ['avg_daily_clicks_prior', 'avg_position_prior', 'click_consistency_prior', 'ai_share_prior', 'no_ranking_data_prior']
honest AUC: 0.633  |  deliberately-leaky AUC: 0.997  (should jump toward 1.0)
no product flags (health_score, needs_ctr_fix, etc.) in feature set: CONFIRMED — none were ever queried from the warehouse
ga4_coverage_prior excluded: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (ML-08): "the Random Forest achieves Precision@50 = 0.78–0.80... genuine signal that
generalizes to clients the model never saw during training."

Rewritten: On a client-grouped holdout split we measured Precision@50 = 0.740 (base rate 0.652),
a directional, decision-support improvement over the position-only baseline's near-base-rate
performance on the same split. The same model scored a perfect 1.000 on a naive random split —
a 26-point gap that is itself the finding: it demonstrates concretely, on our own data, the exact
memorization risk we questioned in the paper's ungrouped growth-classification claim (Finding 2).
No causal language is used; this is observed, out-of-sample ranking performance, not a claim
about what drives decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.